# PathSnap – Person B: Dataset Exploration

**Notebook 01** | Descriptive inspection of permitted development data.

Rules: inspect `train` and `student_test` only. Do not inspect evaluator-testing data during development. Do not assume GPS rows align positionally with route rows. Raw files are whitespace-separated and headerless.

## 1. Imports and configuration

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", 100)
try:
    import networkx as nx
    NETWORKX_AVAILABLE = True
except ImportError:
    NETWORKX_AVAILABLE = False
    print("Install networkx for connectivity analysis: pip install networkx")

# Update this path for your repository layout.
DATA_ROOT = Path("../../data/raw/Student folder")
TRAIN_DIR = DATA_ROOT / "train"
STUDENT_TEST_DIR = DATA_ROOT / "student_test"
EXTENSIONS = [".track", ".nodes", ".arcs", ".route"]

## 2. Discover trips and inspect file availability

In [ ]:
def discover_trips(folder):
    if not folder.exists():
        raise FileNotFoundError(f"Not found: {folder.resolve()}. Update DATA_ROOT.")
    return sorted(p.name for p in folder.iterdir() if p.is_dir())

def file_path(folder, trip_id, ext):
    return folder / trip_id / f"{trip_id}{ext}"

train_ids = discover_trips(TRAIN_DIR)
student_ids = discover_trips(STUDENT_TEST_DIR)
print("Training trips:", len(train_ids))
print("Student-testing trips:", len(student_ids))

def inventory_for(folder, ids, partition):
    rows=[]
    for tid in ids:
        r={"partition":partition,"trip_id":tid}
        for ext in EXTENSIONS:
            p=file_path(folder,tid,ext)
            r[ext[1:]+"_exists"]=p.is_file()
            r[ext[1:]+"_bytes"]=p.stat().st_size if p.is_file() else np.nan
        rows.append(r)
    return pd.DataFrame(rows)
inventory=pd.concat([inventory_for(TRAIN_DIR,train_ids,"train"),inventory_for(STUDENT_TEST_DIR,student_ids,"student_test")],ignore_index=True)
display(inventory.head(10))
display(inventory.groupby("partition")[[e[1:]+"_exists" for e in EXTENSIONS]].sum())

## 3. Read raw files using the documented schemas

In [ ]:
def read_trip(folder, tid):
    p={e:file_path(folder,tid,e) for e in EXTENSIONS}
    missing=[str(v) for v in p.values() if not v.is_file()]
    if missing: raise FileNotFoundError(f"{tid}: missing {missing}")
    return {
        "track":pd.read_csv(p[".track"],sep=r"\s+",header=None,names=["longitude","latitude","timestamp"]),
        "nodes":pd.read_csv(p[".nodes"],sep=r"\s+",header=None,names=["longitude","latitude"]),
        "arcs":pd.read_csv(p[".arcs"],sep=r"\s+",header=None,names=["from_node_id","to_node_id"]),
        "route":pd.read_csv(p[".route"],sep=r"\s+",header=None,names=["arc_id"])}

sample_id=train_ids[0]
sample=read_trip(TRAIN_DIR,sample_id)
for name,df in sample.items():
    print(name, df.shape)
    display(df.head())

## 4. Per-trip quality and size summary

In [ ]:
def haversine_km(lon1,lat1,lon2,lat2):
    R=6371.0088
    lon1,lat1,lon2,lat2=map(np.radians,[lon1,lat1,lon2,lat2])
    a=np.sin((lat2-lat1)/2)**2+np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2
    return 2*R*np.arcsin(np.sqrt(a))

def summarise(folder,tid,partition):
    d=read_trip(folder,tid); t,n,a,r=d["track"],d["nodes"],d["arcs"],d["route"]
    dt=t.timestamp.diff().dropna(); valid=t.longitude.between(-180,180)&t.latitude.between(-90,90)&t.notna().all(axis=1)
    node_ids=set(range(len(n)))
    bad_endpoints=sum(int(u) not in node_ids or int(v) not in node_ids for u,v in a.itertuples(index=False,name=None))
    bad_route=int((~r.arc_id.between(0,len(a)-1)).sum()) if len(a) else len(r)
    step=haversine_km(t.longitude.shift(),t.latitude.shift(),t.longitude,t.latitude)
    return {"partition":partition,"trip_id":tid,"gps_observations":len(t),"nodes":len(n),"arcs":len(a),"route_arcs":len(r),
      "missing_gps_rows":int(t.isna().any(axis=1).sum()),"invalid_gps_rows":int((~valid).sum()),
      "duplicate_timestamps":int(t.timestamp.duplicated().sum()),"non_increasing_steps":int((dt<=0).sum()),
      "non_1s_steps":int((dt!=1).sum()),"max_time_gap_s":float(dt.max()) if len(dt) else np.nan,
      "invalid_arc_endpoints":bad_endpoints,"invalid_route_arc_ids":bad_route,
      "longitude_min":t.longitude.min(),"longitude_max":t.longitude.max(),"latitude_min":t.latitude.min(),"latitude_max":t.latitude.max(),
      "gps_path_length_km":float(step.sum()),
      **{e[1:]+"_bytes":file_path(folder,tid,e).stat().st_size for e in EXTENSIONS}}

rows=[summarise(TRAIN_DIR,x,"train") for x in train_ids]+[summarise(STUDENT_TEST_DIR,x,"student_test") for x in student_ids]
trip_summary=pd.DataFrame(rows)
display(trip_summary.head())
display(trip_summary.groupby("partition").describe().T)

## 5. Timestamp and trajectory plots

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(13,4))
for name,g in trip_summary.groupby("partition"):
    ax[0].hist(g.gps_observations,bins=15,alpha=.5,label=name)
    ax[1].hist(g.max_time_gap_s.dropna(),bins=15,alpha=.5,label=name)
ax[0].set(title="GPS observations per trip",xlabel="Observations")
ax[1].set(title="Maximum timestamp gap",xlabel="Seconds")
for a in ax: a.legend(); a.grid(alpha=.2)
plt.tight_layout(); plt.show()
display(trip_summary[["partition","trip_id","missing_gps_rows","invalid_gps_rows","duplicate_timestamps","non_increasing_steps","non_1s_steps","max_time_gap_s"]].sort_values("max_time_gap_s",ascending=False).head(20))

## 6. Road-network connectivity

In [ ]:
def connectivity(arcs,n_nodes):
    if not NETWORKX_AVAILABLE: return {"status":"networkx unavailable"}
    G=nx.Graph(); G.add_nodes_from(range(n_nodes)); G.add_edges_from(arcs.itertuples(index=False,name=None))
    comps=list(nx.connected_components(G)); largest=max((len(c) for c in comps),default=0)
    return {"components":len(comps),"isolated_nodes":sum(deg==0 for _,deg in G.degree()),
            "largest_component_nodes":largest,"largest_component_fraction":largest/n_nodes if n_nodes else np.nan}

conn=[]
for partition,folder,ids in [("train",TRAIN_DIR,train_ids),("student_test",STUDENT_TEST_DIR,student_ids)]:
    for tid in ids:
        d=read_trip(folder,tid); conn.append({"partition":partition,"trip_id":tid,**connectivity(d["arcs"],len(d["nodes"]))})
connectivity_df=pd.DataFrame(conn)
display(connectivity_df)

## 7. Plot one trip against its road-network nodes

In [ ]:
d=read_trip(TRAIN_DIR,sample_id); tr,nd=d["track"],d["nodes"]
plt.figure(figsize=(8,7))
plt.scatter(nd.longitude,nd.latitude,s=1,alpha=.25,label="Road-network nodes")
plt.plot(tr.longitude,tr.latitude,linewidth=1,marker=".",markersize=2,label="GPS trajectory")
plt.xlabel("Longitude"); plt.ylabel("Latitude"); plt.title(f"Training trip {sample_id}")
plt.axis("equal"); plt.grid(alpha=.2); plt.legend(); plt.show()

## 8. Export descriptive outputs

In [ ]:
OUTPUT_DIR=Path("../../outputs/exploration")
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
inventory.to_csv(OUTPUT_DIR/"file_inventory.csv",index=False)
trip_summary.to_csv(OUTPUT_DIR/"trip_summary.csv",index=False)
connectivity_df.to_csv(OUTPUT_DIR/"connectivity_summary.csv",index=False)
print("Saved outputs to",OUTPUT_DIR.resolve())

## 9. Findings and handoff checklist

Complete these notes after running the notebook; do not invent findings.

- [ ] Confirmed training and student-testing trip counts.
- [ ] Checked raw file availability.
- [ ] Reviewed GPS validity and timestamp gaps.
- [ ] Checked node/arc counts and endpoint references.
- [ ] Checked route arc IDs against arc indices.
- [ ] Reviewed disconnected components and isolated nodes.
- [ ] Shared anomalies with Person A.
- [ ] Confirmed candidate-table schema with Person A.

**Before model training, Person A must confirm candidate generation, observation-to-route alignment, label definition, feature columns and `trip_id`. Never assume GPS row index equals route row index.**